# 02 — Завантаження VNP46A4 і агрегація даних нічного освітлення на рівні країн


Цей ноутбук формує маніфест файлів VNP46A4 для досліджуваної території, перевіряє та завантажує річні тайли, застосовує фінальні маски якості (QA) та маски «суходіл-вода», агрегує статистику нічного освітлення на рівні «країна — рік» і зберігає результати.

## 1. Налаштування середовища та параметрів

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
from collections import defaultdict
import gc
import hashlib
import os
import re

import geopandas as gpd
import h5py
import numpy as np
import pandas as pd
import requests
from google.colab import userdata
from pyproj import Geod
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds
from shapely.geometry import box

PROJECT_ROOT = Path("/content/drive/MyDrive/GDP_NTL_Project")
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
NTL_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ntl"
TEMP_DIR = Path("/content/tmp_vnp46a4")

STUDY_AREA_PATH = METADATA_DIR / "study_area.geojson"
COUNTRY_TILE_PATH = METADATA_DIR / "country_tile_map.csv"
MANIFEST_PATH = METADATA_DIR / "vnp46a4_manifest_2012_2025.csv"
SUMMARY_PATH = NTL_PROCESSED_DIR / "vnp46a4_processing_summary_2012_2025.csv"

for path in [METADATA_DIR, NTL_PROCESSED_DIR, TEMP_DIR]:
    path.mkdir(parents=True, exist_ok=True)

API_URL = "https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/details"
PRODUCT = "VNP46A4"
REFERENCE_YEAR = 2024
YEARS = range(2012, 2026)
DATA_FIELDS = "HDFEOS/GRIDS/VIIRS_Grid_DNB_2d/Data Fields/"
VALID_SURFACE_CLASSES = [0, 1, 5]  # Land & Desert, Land, Coastal
LIT_RADIANCE_THRESHOLD = 0.5  # nW/(cm² sr)
GEOD = Geod(ellps="WGS84")


NASA_TOKEN = userdata.get("NTLDATA_TOKEN")
if not NASA_TOKEN:
    raise ValueError("NTLDATA_TOKEN not found in Colab Secrets.")

## 2. Завантаження геоданих області дослідження і визначення потрібних VNP46A4 тайлів

In [ ]:
study_area = gpd.read_file(STUDY_AREA_PATH)
print("Countries:", study_area["CNTR_ID"].nunique())
print("CRS:", study_area.crs)
study_area.head()

Countries: 36
CRS: EPSG:4326


,CNTR_ID,COUNTRY_URI,CNTR_NAME,NAME_ENGL,NAME_FREN,ISO3_CODE,SVRG_UN,CAPT,STAT_CODE,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry
0,DK,DNK,Danmark,Denmark,Danemark,DNK,UN Member State,Copenhagen,OA,T,F,F,Dänemark,"MULTIPOLYGON (((10.43458 57.59716, 10.43276 57..."
1,EE,EST,Eesti,Estonia,Estonie,EST,UN Member State,Tallinn,OA,T,F,F,Estland,"MULTIPOLYGON (((25.78573 59.57444, 25.83016 59..."
2,AL,ALB,Shqipëria,Albania,Albanie,ALB,UN Member State,Tirana,OA,F,F,T,Albanien,"MULTIPOLYGON (((19.80414 42.47634, 19.83093 42..."
3,AT,AUT,Österreich,Austria,Autriche,AUT,UN Member State,Vienna,OA,T,F,F,Österreich,"MULTIPOLYGON (((15.16631 48.9675, 15.16298 48...."
4,BA,BIH,Bosna I Hercegovina-Bosna I Hercegovina-Босна ...,Bosnia and Herzegovina,Bosnie-et-Herzégovine,BIH,UN Member State,Sarajevo,OA,F,F,T,Bosnien und Herzegowina,"MULTIPOLYGON (((16.9429 45.23201, 16.96709 45...."


In [ ]:
def calculate_geometry_area_km2(geometry):
    area_m2, _ = GEOD.geometry_area_perimeter(geometry)
    return abs(area_m2) / 1_000_000

study_area["country_area_km2"] = study_area.geometry.apply(calculate_geometry_area_km2)

In [ ]:
def get_country_tiles(country_row, reference_year=REFERENCE_YEAR):
    geometry = country_row.geometry
    parts = list(geometry.geoms) if geometry.geom_type == "MultiPolygon" else [geometry]
    records = []

    for part in parts:
        west, south, east, north = part.bounds
        params = {
            "products": PRODUCT,
            "temporalRanges": f"{reference_year}-01-01..{reference_year}-12-31",
            "regions": f"[BBOX]W{west} N{north} E{east} S{south}",
        }
        response = requests.get(API_URL, params=params, timeout=120)
        response.raise_for_status()

        for file in response.json().get("content", []):
            match = re.search(r"\.(h\d{2}v\d{2})\.", file.get("name", ""))
            if match:
                records.append({
                    "tile": match.group(1),
                    "west": file["westBoundCoord"],
                    "south": file["southBoundCoord"],
                    "east": file["eastBoundCoord"],
                    "north": file["northBoundCoord"],
                    "size_bytes": file["size"],
                })
    return records

all_records = []
for _, country in study_area.iterrows():
    all_records.extend(get_country_tiles(country))

candidate_tiles = pd.DataFrame(all_records).drop_duplicates(subset="tile")
candidate_tiles["geometry"] = candidate_tiles.apply(
    lambda r: box(r["west"], r["south"], r["east"], r["north"]), axis=1
)
candidate_tiles = gpd.GeoDataFrame(candidate_tiles, geometry="geometry", crs="EPSG:4326")

country_tile_map = gpd.sjoin(
    study_area[["CNTR_ID", "NAME_ENGL", "geometry"]],
    candidate_tiles[["tile", "size_bytes", "geometry"]],
    how="inner",
    predicate="intersects",
)[["CNTR_ID", "NAME_ENGL", "tile", "size_bytes"]].drop_duplicates()

country_tile_map = country_tile_map.sort_values(["CNTR_ID", "tile"]).reset_index(drop=True)
required_tiles = sorted(country_tile_map["tile"].unique())

country_tile_map[["CNTR_ID", "NAME_ENGL", "tile"]].to_csv(COUNTRY_TILE_PATH, index=False)
print("Required tiles:", len(required_tiles))
print(required_tiles)

KeyboardInterrupt: 

## 3. Створення та валідація маніфесту даних

In [ ]:
def build_vnp46a4_manifest(years, required_tiles):
    records = []
    required_tiles = set(required_tiles)

    for requested_year in years:
        print(f"Checking {requested_year}...")
        params = {
            "products": PRODUCT,
            "temporalRanges": f"{requested_year}-01-01..{requested_year}-12-31",
        }
        response = requests.get(API_URL, params=params, timeout=120)
        response.raise_for_status()
        first_page = response.json()

        for page in range(1, first_page.get("n_pages", 1) + 1):
            if page == 1:
                page_data = first_page
            else:
                page_params = {**params, "page": page}
                page_response = requests.get(API_URL, params=page_params, timeout=120)
                page_response.raise_for_status()
                page_data = page_response.json()

            for file in page_data.get("content", []):
                match = re.search(r"\.A(\d{4})\d{3}\.(h\d{2}v\d{2})\.", file.get("name", ""))
                if not match:
                    continue
                actual_year, tile = int(match.group(1)), match.group(2)
                if actual_year == requested_year and tile in required_tiles:
                    records.append({
                        "year": actual_year,
                        "tile": tile,
                        "filename": file.get("name"),
                        "size_bytes": file.get("size"),
                        "download_url": file.get("downloadsLink"),
                        "md5": file.get("md5sum"),
                        "status": file.get("status"),
                    })

    return pd.DataFrame(records).sort_values(["year", "tile"]).reset_index(drop=True)

availability_ntl = build_vnp46a4_manifest(YEARS, required_tiles)
availability_ntl.to_csv(MANIFEST_PATH, index=False)
print("Manifest files:", len(availability_ntl))

In [ ]:
expected = pd.MultiIndex.from_product([YEARS, required_tiles], names=["year", "tile"]).to_frame(index=False)
actual = availability_ntl[["year", "tile"]].drop_duplicates()
missing_files = expected.merge(actual, on=["year", "tile"], how="left", indicator=True)
missing_files = missing_files.loc[missing_files["_merge"] == "left_only", ["year", "tile"]]

duplicates = (availability_ntl.groupby(["year", "tile"]).size().reset_index(name="n_files").query("n_files > 1"))
summary_by_year = (availability_ntl.groupby("year")["tile"].nunique().reindex(YEARS, fill_value=0).rename("available_tiles").reset_index())
summary_by_year["expected_tiles"] = len(required_tiles)
summary_by_year["missing_tiles"] = summary_by_year["expected_tiles"] - summary_by_year["available_tiles"]

print("Expected files:", len(expected))
print("Found files:", len(availability_ntl))
print("Missing files:", len(missing_files))
print("Duplicate year × tile combinations:", len(duplicates))
print(f"Total raw volume: {availability_ntl['size_bytes'].sum() / 1024**3:.2f} GB")
display(summary_by_year)
if not missing_files.empty:
    display(missing_files)
if not duplicates.empty:
    display(duplicates)

## 4. Завантаження і читання VNP46A4 тайлів

In [ ]:
def download_vnp46a4_file(file_info, nasa_token):
    local_path = TEMP_DIR / file_info["filename"]
    headers = {"Authorization": f"Bearer {nasa_token}"}

    with requests.get(file_info["download_url"], headers=headers, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    with open(local_path, "rb") as f:
        if f.read(8) != b"\x89HDF\r\n\x1a\n":
            local_path.unlink(missing_ok=True)
            raise ValueError(f"{file_info['filename']} is not a valid HDF5 file.")

    if local_path.stat().st_size != int(file_info["size_bytes"]):
        local_path.unlink(missing_ok=True)
        raise ValueError(f"File size mismatch: {file_info['filename']}")

    md5 = hashlib.md5()
    with open(local_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            md5.update(chunk)
    if md5.hexdigest().lower() != str(file_info["md5"]).lower():
        local_path.unlink(missing_ok=True)
        raise ValueError(f"MD5 mismatch: {file_info['filename']}")

    return local_path

def tile_bounds(tile):
    h, v = int(tile[1:3]), int(tile[4:6])
    west = -180 + h * 10
    north = 90 - v * 10
    return west, north - 10, west + 10, north

def read_vnp46a4_tile(file_path, west, south, east, north):
    with h5py.File(file_path, "r") as h5:
        rad_ds = h5[DATA_FIELDS + "NearNadir_Composite_Snow_Free"]
        radiance_raw = rad_ds[:].astype(np.float32)
        rad_fill = rad_ds.attrs.get("_FillValue", -999.9)

        quality_ds = h5[DATA_FIELDS + "NearNadir_Composite_Snow_Free_Quality"]
        quality = quality_ds[:]
        quality_fill = quality_ds.attrs.get("_FillValue", 255)

        nobs_ds = h5[DATA_FIELDS + "NearNadir_Composite_Snow_Free_Num"]
        n_obs = nobs_ds[:].astype(np.float32)
        nobs_fill = nobs_ds.attrs.get("_FillValue", 65535)

        surface_ds = h5[DATA_FIELDS + "Land_Water_Mask"]
        surface = surface_ds[:]
        surface_fill = surface_ds.attrs.get("_FillValue", 255)

    valid_radiance = ~np.isclose(radiance_raw, rad_fill) & (radiance_raw >= 0)
    valid_quality = quality == 0
    valid_surface = np.isin(surface, VALID_SURFACE_CLASSES)
    valid_mask = valid_radiance & valid_quality & valid_surface

    radiance = radiance_raw.copy()
    radiance[~valid_mask] = np.nan
    n_obs[n_obs == nobs_fill] = np.nan
    n_obs[~valid_mask] = np.nan

    height, width = radiance.shape
    transform = from_bounds(west, south, east, north, width, height)

    return {
        "radiance": radiance, "n_obs": n_obs,
        "radiance_raw": radiance_raw, "rad_fill": rad_fill,
        "quality": quality, "quality_fill": quality_fill,
        "surface": surface, "surface_fill": surface_fill,
        "transform": transform, "bounds": (west, south, east, north),
    }

## 5. Просторове виділення та агрегація даних за країнами

In [ ]:
def get_pixel_area_by_row(transform, height, width):
    pixel_width = transform.a
    pixel_height = abs(transform.e)
    areas = np.empty(height, dtype=np.float64)
    for row in range(height):
        north = transform.f + row * transform.e
        south = north - pixel_height
        west, east = transform.c, transform.c + pixel_width
        area_m2, _ = GEOD.polygon_area_perimeter(
            [west, east, east, west], [north, north, south, south]
        )
        areas[row] = abs(area_m2) / 1_000_000
    return areas

def extract_country_pixels(tile_data, country_geometry, quality_values=(0,)):
    raw = tile_data["radiance_raw"]
    quality = tile_data["quality"]
    surface = tile_data["surface"]
    transform = tile_data["transform"]
    height, width = raw.shape

    country_mask = geometry_mask([country_geometry], out_shape=(height, width), transform=transform, invert=True, all_touched=False)
    valid_radiance = ~np.isclose(raw, tile_data["rad_fill"]) & (raw >= 0)
    valid = country_mask & valid_radiance & np.isin(quality, quality_values) & np.isin(surface, VALID_SURFACE_CLASSES)
    if not np.any(valid):
        return None

    rows, cols = np.where(valid)
    row_area = get_pixel_area_by_row(transform, height, width)
    result = {"radiance": raw[rows, cols], "area_km2": row_area[rows]}
    if quality_values == (0,):
        result["n_obs"] = tile_data["n_obs"][rows, cols]
    return result

def aggregate_country_pixels(
    pixel_parts,
    include_nobs=True,
    lit_threshold=LIT_RADIANCE_THRESHOLD
):
    parts = [p for p in pixel_parts if p is not None]

    if not parts:
        return None

    radiance = np.concatenate([
        p["radiance"] for p in parts
    ])

    area = np.concatenate([
        p["area_km2"] for p in parts
    ])

    # -----------------------------------------------------
    # Основні статистики розподілу NTL
    # -----------------------------------------------------
    ntl_mean = np.mean(radiance)
    ntl_std = np.std(radiance)
    ntl_p25 = np.percentile(radiance, 25)
    ntl_median = np.median(radiance)
    ntl_p75 = np.percentile(radiance, 75)
    ntl_p90 = np.percentile(radiance, 90)
    ntl_p95 = np.percentile(radiance, 95)

    ntl_iqr = ntl_p75 - ntl_p25

    if ntl_mean > 0:
        ntl_cv = ntl_std / ntl_mean
    else:
        ntl_cv = np.nan

    # -----------------------------------------------------
    # Area-weighted NTL
    # -----------------------------------------------------
    valid_ntl_area_km2 = np.sum(area)

    ntl_mean_area_weighted = np.average(
        radiance,
        weights=area
    )

    ntl_sum_area_weighted = np.sum(
        radiance * area
    )

    # -----------------------------------------------------
    # Освітлена територія
    # -----------------------------------------------------
    lit_mask = radiance >= lit_threshold

    lit_area_km2 = np.sum(area[lit_mask])

    if valid_ntl_area_km2 > 0:
        lit_area_share_valid = (
            lit_area_km2 / valid_ntl_area_km2
        )
    else:
        lit_area_share_valid = np.nan

    if np.any(lit_mask):
        ntl_positive_mean = np.mean(
            radiance[lit_mask]
        )
    else:
        ntl_positive_mean = np.nan

    # -----------------------------------------------------
    # Результат
    # -----------------------------------------------------
    out = {
        "n_valid_pixels": int(radiance.size),
        "valid_ntl_area_km2": float(valid_ntl_area_km2),

        "ntl_mean": float(ntl_mean),
        "ntl_median": float(ntl_median),
        "ntl_std": float(ntl_std),

        "ntl_p25": float(ntl_p25),
        "ntl_p75": float(ntl_p75),
        "ntl_p90": float(ntl_p90),
        "ntl_p95": float(ntl_p95),

        "ntl_iqr": float(ntl_iqr),
        "ntl_cv": float(ntl_cv),

        "ntl_mean_area_weighted": float(
            ntl_mean_area_weighted
        ),

        "ntl_sum_area_weighted": float(
            ntl_sum_area_weighted
        ),

        "lit_area_km2": float(lit_area_km2),
        "lit_area_share_valid": float(
            lit_area_share_valid
        ),

        "ntl_positive_mean": float(
            ntl_positive_mean
        ),
    }

    # -----------------------------------------------------
    # Кількість спостережень VIIRS
    # -----------------------------------------------------
    if include_nobs:
        n_obs = np.concatenate([
            p["n_obs"] for p in parts
        ])

        out["mean_n_observations"] = float(
            np.nanmean(n_obs)
        )

    return out

def extract_country_qa(tile_data, country_geometry):
    raw, quality, surface = tile_data["radiance_raw"], tile_data["quality"], tile_data["surface"]
    transform = tile_data["transform"]
    height, width = raw.shape
    country_mask = geometry_mask([country_geometry], out_shape=(height, width), transform=transform, invert=True, all_touched=False)
    valid_radiance = ~np.isclose(raw, tile_data["rad_fill"]) & (raw >= 0)
    valid_surface = np.isin(surface, VALID_SURFACE_CLASSES)
    row_area = get_pixel_area_by_row(transform, height, width)

    masks = {
        "water_km2": country_mask & np.isin(surface, [2, 3]),
        "qa0_km2": country_mask & valid_surface & valid_radiance & (quality == 0),
        "qa1_km2": country_mask & valid_surface & valid_radiance & (quality == 1),
        "qa2_km2": country_mask & valid_surface & valid_radiance & (quality == 2),
        "invalid_radiance_km2": country_mask & valid_surface & ~valid_radiance,
    }
    return {name: float(row_area[np.where(mask)[0]].sum()) if np.any(mask) else 0.0 for name, mask in masks.items()}

## 6. Обробка одного року



In [ ]:
def process_ntl_year(year, save=True):
    files = availability_ntl.loc[availability_ntl["year"] == year].sort_values("tile").reset_index(drop=True)
    if files.empty:
        raise ValueError(f"No VNP46A4 files found for {year}.")

    qa0_parts = defaultdict(list)
    qa01_parts = defaultdict(list)
    qa_parts = defaultdict(list)

    for i, file_info in files.iterrows():
        tile = file_info["tile"]
        print(f"[{i + 1}/{len(files)}] {tile}")
        local_path = None
        tile_data = None
        try:
            local_path = download_vnp46a4_file(file_info, NASA_TOKEN)
            tile_data = read_vnp46a4_tile(local_path, *tile_bounds(tile))
            countries = country_tile_map.loc[country_tile_map["tile"] == tile, "CNTR_ID"].unique()

            for code in countries:
                geometry = study_area.loc[study_area["CNTR_ID"] == code, "geometry"].iloc[0]
                qa0_parts[code].append(extract_country_pixels(tile_data, geometry, (0,)))
                qa01_parts[code].append(extract_country_pixels(tile_data, geometry, (0, 1)))
                qa_parts[code].append(extract_country_qa(tile_data, geometry))
        finally:
            if local_path is not None:
                Path(local_path).unlink(missing_ok=True)
            del tile_data
            gc.collect()

    rows = []
    for _, country in study_area.iterrows():
        code = country["CNTR_ID"]
        main = aggregate_country_pixels(qa0_parts[code], include_nobs=True)
        alt = aggregate_country_pixels(qa01_parts[code], include_nobs=False)
        if main is None:
            continue

        qa_sum = {k: sum(part[k] for part in qa_parts[code]) for k in ["water_km2", "qa0_km2", "qa1_km2", "qa2_km2", "invalid_radiance_km2"]}
        area = country["country_area_km2"]
        row = {
            "CNTR_ID": code, "NAME_ENGL": country["NAME_ENGL"], "year": year,
            "country_area_km2": area, **main,
            "ntl_coverage_share": main["valid_ntl_area_km2"] / area,
            **qa_sum,
        }
        for key in qa_sum:
            row[key.replace("_km2", "_share")] = qa_sum[key] / area

        if alt is not None:
            row.update({
                "valid_ntl_area_km2_qa01": alt["valid_ntl_area_km2"],
                "ntl_mean_area_weighted_qa01": alt["ntl_mean_area_weighted"],
                "ntl_sum_area_weighted_qa01": alt["ntl_sum_area_weighted"],
                "ntl_coverage_share_qa01": alt["valid_ntl_area_km2"] / area,
            })
        rows.append(row)

    result = pd.DataFrame(rows).sort_values("CNTR_ID").reset_index(drop=True)
    if save:
        output_path = NTL_PROCESSED_DIR / f"vnp46a4_country_ntl_{year}.csv"
        result.to_csv(output_path, index=False)
        print("Saved:", output_path)
    return result

## 7. QA перевірка одного року

In [ ]:
QA_YEAR = 2023
qa_year = process_ntl_year(QA_YEAR, save=False)
qa_view = qa_year[[
    "CNTR_ID", "NAME_ENGL", "ntl_coverage_share", "ntl_coverage_share_qa01",
    "water_share", "qa0_share", "qa1_share", "qa2_share", "invalid_radiance_share"
]].sort_values("ntl_coverage_share")
display(qa_view.head(15))

## 8. Обробка всіх років та збереження результатів

In [ ]:
processing_summary = []

for year in YEARS:
    print(f"\n{'#' * 60}\nYEAR: {year}\n{'#' * 60}")
    try:
        result_year = process_ntl_year(year, save=True)
        processing_summary.append({
            "year": year, "status": "success", "n_countries": len(result_year),
            "min_qa0_coverage": result_year["ntl_coverage_share"].min(),
            "min_qa01_coverage": result_year["ntl_coverage_share_qa01"].min(),
            "max_qa1_share": result_year["qa1_share"].max(),
            "max_qa2_share": result_year["qa2_share"].max(), "error": None,
        })
    except Exception as exc:
        print(f"Error in {year}: {exc}")
        processing_summary.append({
            "year": year, "status": "error", "n_countries": None,
            "min_qa0_coverage": None, "min_qa01_coverage": None,
            "max_qa1_share": None, "max_qa2_share": None, "error": str(exc),
        })
        gc.collect()

processing_summary = pd.DataFrame(processing_summary)
processing_summary.to_csv(SUMMARY_PATH, index=False)
display(processing_summary)
print("Successful years:", (processing_summary["status"] == "success").sum())
print("Years with errors:", (processing_summary["status"] == "error").sum())
print("Summary saved:", SUMMARY_PATH)